# 04 — Redemption: The 2024 T20 World Cup

India won the T20 World Cup on June 29, 2024, defeating South Africa. After the 2023 WC heartbreak, this was the redemption arc. This notebook analyzes the Reddit reaction and contrasts it with the 2023 loss.

### 1. Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from utils import (
    get_spark, load_cricket_submissions, load_cricket_comments,
    add_player_mentions, add_time_features, label_event_period,
    save_figure, save_pandas_to_local,
    EVENT_DATES,
)

from pyspark.sql import functions as F
from pyspark.sql.functions import col, count, avg, sum as ssum, when, lower, to_date
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd

sns.set_theme(style="darkgrid")
plt.rcParams["figure.dpi"] = 120

### 2. Start Spark

In [ ]:
spark = get_spark("04_Redemption2024T20")
spark

### 3. Load filtered data

In [ ]:
subs = load_cricket_submissions(spark)
coms = load_cricket_comments(spark)
print("Loaded submissions and comments")

### 4. Add time features and event period labels

In [ ]:
subs = add_time_features(subs, ts_col="created_utc")
subs = label_event_period(subs, ts_col="created_dt")

coms = add_time_features(coms, ts_col="created_utc")
coms = label_event_period(coms, ts_col="created_dt")

### 5. Narrow to T20 WC 2024 window

In [ ]:
t20_subs = subs.filter(
    (col("created_dt") >= EVENT_DATES["t20wc2024_start"]) &
    (col("created_dt") <= EVENT_DATES["t20wc2024_end"])
)
t20_coms = coms.filter(
    (col("created_dt") >= EVENT_DATES["t20wc2024_start"]) &
    (col("created_dt") <= EVENT_DATES["t20wc2024_end"])
)
print(f"T20 WC 2024 submissions: {t20_subs.count():,}")
print(f"T20 WC 2024 comments:    {t20_coms.count():,}")

### 6. Daily post volume

In [ ]:
daily_posts = (
    t20_subs
    .withColumn("date", to_date(col("created_dt")))
    .groupBy("date")
    .agg(count("*").alias("post_count"))
    .orderBy("date")
    .toPandas()
)
daily_posts["date"] = pd.to_datetime(daily_posts["date"])
print(daily_posts)

### 7. Line chart — daily post volume

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_posts["date"], daily_posts["post_count"], color="#e9c46a", linewidth=2, marker="o", markersize=4)
ax.axvline(EVENT_DATES["t20wc2024_final"], color="gold", linestyle="--", linewidth=1.5, label="Final: India vs South Africa (Win!)")
ax.fill_between(daily_posts["date"], daily_posts["post_count"], alpha=0.15, color="#e9c46a")
ax.set_title("Daily Reddit Post Volume — 2024 T20 World Cup", fontsize=13)
ax.set_ylabel("Posts per Day")
ax.set_xlabel("Date")
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.xticks(rotation=30)
plt.tight_layout()
save_figure(fig, "04_daily_posts_t20wc2024.png")
plt.show()

### 8. Player mentions during T20 WC 2024

In [ ]:
t20_subs = add_player_mentions(t20_subs, text_col="title")
t20_coms = add_player_mentions(t20_coms, text_col="body")

t20_mentions = t20_subs.agg(
    ssum(col("mentions_dhoni").cast("int")).alias("Dhoni"),
    ssum(col("mentions_kohli").cast("int")).alias("Kohli"),
    ssum(col("mentions_rohit").cast("int")).alias("Rohit"),
).toPandas()
print("Player mentions during T20 WC 2024:")
print(t20_mentions)

### 9. Player mentions by period

In [ ]:
mentions_by_period = t20_subs.groupBy("event_period").agg(
    ssum(col("mentions_dhoni").cast("int")).alias("Dhoni"),
    ssum(col("mentions_kohli").cast("int")).alias("Kohli"),
    ssum(col("mentions_rohit").cast("int")).alias("Rohit"),
).toPandas()
print(mentions_by_period)
save_pandas_to_local(mentions_by_period, "04_t20wc2024_mentions_by_period.csv")

### 10. Grouped bar — mentions by period

In [ ]:
plot_df = mentions_by_period.set_index("event_period")[["Dhoni", "Kohli", "Rohit"]]
fig, ax = plt.subplots(figsize=(10, 5))
plot_df.plot(kind="bar", ax=ax, color=["#f4a261", "#2a9d8f", "#e76f51"], edgecolor="white", width=0.6)
ax.set_title("Player Mentions by Event Period — 2024 T20 WC", fontsize=13)
ax.set_ylabel("Mentions")
ax.set_xlabel("")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
ax.legend(title="Player")
plt.tight_layout()
save_figure(fig, "04_t20wc2024_mentions_by_period.png")
plt.show()

### 11. Top posts after the T20 final

In [ ]:
top_aftermath = (
    t20_subs
    .filter(col("event_period") == "t20wc2024_final_aftermath")
    .select("title", "subreddit", "score", "num_comments", "created_dt")
    .orderBy(col("score").desc())
    .limit(10)
    .toPandas()
)
print(top_aftermath[["title", "subreddit", "score", "num_comments"]].to_string(index=False))
save_pandas_to_local(top_aftermath, "04_t20wc2024_top_aftermath_posts.csv")

### 12. Head-to-head: 2023 WC loss vs 2024 T20 win

In [ ]:
wc2023_aftermath = subs.filter(col("event_period") == "wc2023_final_aftermath")
wc2023_stats = wc2023_aftermath.agg(
    count("*").alias("posts"),
    avg("score").alias("avg_score"),
    avg("num_comments").alias("avg_comments"),
).toPandas()
wc2023_stats["event"] = "2023 WC Final (Loss)"

t20_aftermath = subs.filter(col("event_period") == "t20wc2024_final_aftermath")
t20_stats = t20_aftermath.agg(
    count("*").alias("posts"),
    avg("score").alias("avg_score"),
    avg("num_comments").alias("avg_comments"),
).toPandas()
t20_stats["event"] = "2024 T20 Final (Win)"

comparison = pd.concat([wc2023_stats, t20_stats], ignore_index=True)
print(comparison)
save_pandas_to_local(comparison, "04_heartbreak_vs_redemption.csv")

### 13. Bar chart — heartbreak vs redemption

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
event_colors = ["#e76f51", "#2a9d8f"]
metrics = [("posts", "Total Posts"), ("avg_score", "Avg Post Score"), ("avg_comments", "Avg Comments")]

for ax, (metric, label) in zip(axes, metrics):
    ax.bar(comparison["event"], comparison[metric], color=event_colors, edgecolor="white", width=0.5)
    ax.set_title(label)
    ax.set_xticklabels(comparison["event"], rotation=15, ha="right")

plt.suptitle("Reddit Aftermath: Heartbreak 2023 vs Redemption 2024", fontsize=13)
plt.tight_layout()
save_figure(fig, "04_heartbreak_vs_redemption.png")
plt.show()

### 14. Stop Spark

In [ ]:
spark.stop()